In [0]:
# Define Ingestion Parameters & File Map
from pyspark.sql.functions import current_timestamp, input_file_name, count, col

BASE_RAW_PATH = "/Volumes/workspace/olist_raw/raw/"

raw_datasets = {
    "orders": f"{BASE_RAW_PATH}/orders/olist_orders_dataset.csv",
    "customers": f"{BASE_RAW_PATH}/customers/olist_customers_dataset.csv",
    "order_items": f"{BASE_RAW_PATH}/items/olist_order_items_dataset.csv",
    "products": f"{BASE_RAW_PATH}/products/olist_products_dataset.csv",
    "sellers": f"{BASE_RAW_PATH}/sellers/olist_sellers_dataset.csv",
    "payments": f"{BASE_RAW_PATH}/payments/olist_order_payments_dataset.csv",
    "reviews": f"{BASE_RAW_PATH}/reviews/olist_order_reviews_dataset.csv",
    "translations": f"{BASE_RAW_PATH}/translations/product_category_name_translation.csv"
}

In [0]:
# Ingestion & Validation Loop
dataframes = {}
ingestion_summary = []

for entity, file_path in raw_datasets.items():
    print(f"Reading entity: {entity} from {file_path}...")
    
    # Read CSV file into PySpark DataFrame
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .option("multiLine", "true")  # Reviews data often contains multi-line text
          .option("escape", '"')
          .csv(file_path))
    
    # Store DataFrame in dict for interactive access
    dataframes[entity] = df
    
    # Register global temp view for SQL validation queries if needed
    df.createOrReplaceTempView(f"raw_{entity}")
    
    # Record metadata
    record_count = df.count()
    ingestion_summary.append((entity, record_count, len(df.columns)))
    
    print(f"Successfully loaded '{entity}' | Total Records: {record_count:,} | Columns: {len(df.columns)}")

In [0]:
# Display Ingestion Summary Matrix
summary_df = spark.createDataFrame(ingestion_summary, ["Entity", "Record_Count", "Column_Count"])
display(summary_df)

In [0]:
# Validate Schemas and Quick Sample Data
print("=== ORDERS SCHEMA ===")
dataframes["orders"].printSchema()
display(dataframes["orders"].limit(5))

print("=== CUSTOMERS SCHEMA ===")
dataframes["customers"].printSchema()
display(dataframes["customers"].limit(5))

print("=== ORDER_ITEMS SCHEMA ===")
dataframes["order_items"].printSchema()
display(dataframes["order_items"].limit(5))

print("=== PRODUCTS SCHEMA ===")
dataframes["products"].printSchema()
display(dataframes["products"].limit(5))

print("=== SELLERS SCHEMA ===")
dataframes["sellers"].printSchema()
display(dataframes["sellers"].limit(5))

print("=== PAYMENTS SCHEMA ===")
dataframes["payments"].printSchema()
display(dataframes["payments"].limit(5))

print("=== REVIEWS SCHEMA ===")
dataframes["reviews"].printSchema()
display(dataframes["reviews"].limit(5))

print("=== TRANSLATIONS SCHEMA ===")
dataframes["translations"].printSchema()
display(dataframes["translations"].limit(5))